# Baseball Attendance

I think there are a few things that can be very interesting when looking at the attendance data for MLB games:
- Who the opponent is
- Date time information like day of the week, month, or game start time.
- If the team is in playoff contention

### The Data

Baseball attendance data is available in a couple different places. [Retorsheet](https://retrosheet.org/) is great for almost any stats we want, including attendance data. However, it misses a few of extra points I want to explore, like the number of game number or the number of games back in the standings a team is at any given point. These numbers could be calculated ourselves, but for simplicity I'm going to download the data from [Baseball Reference](https://www.baseball-reference.com/) instead.

[Here's a sample](https://www.baseball-reference.com/teams/WSN/2019-schedule-scores.shtml) of what a team's schedule looks like on Baseball Reference. There are several columns that will give more insight into attendance trends:

- `Date`
- `Tm`: The team the data was scraped from
- `Unnamed: 4`: `NaN` or `@` to denote the home team
- `Opp`: The opposing team
- `Rank`: Where the team is in the standings
- `GB`: How many games behind first place the team is in the standings
- `D/N`: Denote a day or night game
- `Attendance`: The number of tickets officially distributed (NOTE: this is *not* how many fans actually attending the game)
- `cLI`: Championship Leverage Index, metric for how important the game is in order to win the championship ([further reading](https://www.sports-reference.com/blog/2020/09/__trashed-2/))
- `Orig. Scheduled`: When the game was originally scheduled. Also provides other information if a game was played at a different location.

#### Baseball Reference

Baseball Reference is a great place to view data for baseball, however it can be a little challenging downloading data from them. First, they don't have any public datasets or an API available, so data needs to be scraped from their site. Second, scraping data from their site can be tedious because they rate limit requests to about 20 requests per minute.

I've included the full raw data in this repo (`raw_output.csv`) so you don't need to wait for the scraping to run, but if you want to download the data yourself you can run the cell below to fetch all of the data. To minimize waiting time, you can also adjust the `start_year` and `end_year`, to focus only on the years you'd like to view. Running the scraping script takes about 75 minutes.

In [ ]:
from data_helper import DataHelper
DataHelper().collect_data()
# DataHelper(start_year=2025, end_year=2025).collect_data()

Using start_year=2025
Using end_year=2025
Created DataHelper!
Finished Diamondbacks
Finished Athletics
Finished Braves
Finished Orioles
Finished Red Sox
Finished Cubs
Finished White Sox
Finished Reds
Finished Guardians
Finished Rockies
Finished Tigers
Finished Astros
Finished Royals
Finished Angels
Finished Dodgers
Finished Marlins
Finished Brewers
Finished Twins
Finished Mets
Finished Yankees
Finished Phillies
Finished Pirates
Finished Padres
Finished Giants
Finished Mariners
Finished Cardinals
Finished Rays
Finished Rangers
Finished Blue Jays
Finished Nationals
Finished Collecting All Data!


### Load the Data

In [71]:
import pandas as pd

df = pd.read_csv('raw_output.csv')
# df.head()

### Cleaning the Data

First, there are several columns that can be removed:

In [72]:
df = df.drop(columns=['Gm#', 'W/L', 'W-L', 'R', 'RA', 'Inn', 'Win', 'Loss', 'Save', 'Time', 'Streak'])
# df.head()

There are two specific row-by-row cases that need to be accounted for:

 First, by default Baseball Reference repeats its column headers on tables at each new month to make it easier to view the data in the browser. Those rows each need to be filtered out.
 
 Second is strange issues coming from teams with cancelled/relocated seasons. For example, the 2025 Athletics played [this schedule](https://www.baseball-reference.com/teams/ATH/2025-schedule-scores.shtml), but were originally scheduled to play [this schedule](https://www.baseball-reference.com/teams/OAK/2025-schedule-scores.shtml) before they relocated from Oakland to Sacramento [on their way to Las Vegas](https://en.wikipedia.org/wiki/Oakland_Athletics_relocation_to_Las_Vegas).
 
 Luckily, we can just filter out both of these types of rows by only selecting for rows where `Unnamed: 2` is "boxscore", representing a game which was played and completed.

In [73]:
df = df[df['Unnamed: 2'] == 'boxscore'].drop(columns=['Unnamed: 2'])
# df.head()

Next, there are a lot of duplicate rows in the data set. Because of the way the data was pulled, an entry is made for both teams for each game. For example:

| Date | Tm | Unnamed: 4 | Opp | ... |
| ---- | -- | ---------- | --- | --- |
| Tuesday, Apr 4 | ARI | NaN | PHI | ... |
| Tuesday, Apr 4 | PHI | @ | ARI | ... |

For the Philidelphia Phillies @ Arizona Diamondbacks game on April 4, 2000.

Fixing this requires only looking at rows where `Unnamed: 4` is `NaN`, basically just selecting for home games, based on `Tm`:

In [74]:
df = df[df['Unnamed: 4'].isna()].drop(columns=['Unnamed: 4'])
# df.head()

At this point, the average number of games each team plays should be approximately 81. Sometimes this number will vary, based on cancelled games or relocated games, but in a typical season this will be exactly 81, like in 2025:

In [75]:
print(f'Home games per team: {len(df[df['year'] == 2025]) / 30}')

Home games per team: 81.0


Next, the `Date` field should be stored as a pandas datetime field for graphing purposes later:

In [76]:
# strip any double header information from the raw date string (example: Thursday, Apr 4 (1))
df['Date'] = df['Date'].str.replace(r'\s*\(\d+\)$', '', regex=True)
df['date'] = pd.to_datetime(df['Date'] + ' ' + df['year'].astype(str), format='%A, %b %d %Y')
df = df.drop(columns=['Date'])
df.head()

,Tm,Opp,Rank,GB,D/N,Attendance,cLI,Orig. Scheduled,year,date
0,ARI,PHI,2,0.5,N,44298,.97,NaN,2000,2000-04-04
1,ARI,PHI,1,up 0.5,N,29291,.97,NaN,2000,2000-04-05
2,ARI,PHI,1,up 1.0,N,28774,1.02,NaN,2000,2000-04-06
3,ARI,PIT,1,Tied,N,32536,1.07,NaN,2000,2000-04-07
4,ARI,PIT,1,up 1.0,D,33298,1.02,NaN,2000,2000-04-08


The GB column can also have some mixed data types since the raw data has values like "up 0.5" and "Tied". Also, since these will be graphed, let's convert it into a more graph friendly format; "up 0.5" becomes `0.5`, "Tied" becomes `0`, and 2.5 (two and a half back) becomes `-2.5`.

In [77]:
import re

def parse_gb(row):
    row = str(row).strip()
    # tied with lead = 0
    if row.lower() == 'tied':
        return 0.0
    # some GB columns come through as up10.5 instead of up 10.5, regex the games out
    match = re.match(r'up\s*([\d.]+)', row, re.IGNORECASE)
    if match:
        return float(match.group(1))
    
    return -float(row)

df['GB'] = df['GB'].apply(parse_gb)
df.head()

,Tm,Opp,Rank,GB,D/N,Attendance,cLI,Orig. Scheduled,year,date
0,ARI,PHI,2,-0.5,N,44298,.97,NaN,2000,2000-04-04
1,ARI,PHI,1,0.5,N,29291,.97,NaN,2000,2000-04-05
2,ARI,PHI,1,1.0,N,28774,1.02,NaN,2000,2000-04-06
3,ARI,PIT,1,0.0,N,32536,1.07,NaN,2000,2000-04-07
4,ARI,PIT,1,1.0,D,33298,1.02,NaN,2000,2000-04-08


There's one last scenario to account for: games with no attendance. This can happen for several reasons, but most notably occured during the COVID-19 pandemic. Filtering out rows with no attendance will solve that:

In [78]:
df = df[df['Attendance'].notna()]
# df.head()

Renaming the columns to something more readable and correcting the data types will also be helpful:

In [79]:
df = df.rename(columns={
    'Tm': 'home_team',
    'Opp': 'away_team',
    'Rank': 'rank',
    'GB': 'standings_gap',
    'D/N': 'day_night',
    'Attendance': 'attendance',
    'Orig. Scheduled': 'orig_scheduled'
})

df = df.astype({
    'rank': 'int8',
    'attendance': 'int32',
    'cLI': 'float32'
})

# df.head()

And lastly, we'll need a game number for each team, denoting which home game it is for that season. This makes it easier to plot later and compare teams, since teams don't necessarily always play on the same dates.

In [80]:
df = df.sort_values('date')
df['game_number'] = df.groupby(['home_team', 'year']).cumcount() + 1

Lastly, let's save the processed dataframe to refer to later.

In [81]:
df.to_csv('processed_data.csv')

## Dashboards

In [ ]:
from dash import Dash, html, dcc, callback, Output, Input
import plotly.express as px
import pandas as pd
from team_helper import get_team_colors, TEAM_NAMES

# load the data if starting from here
try:
    df == None
except NameError:
    df = pd.read_csv('processed_data.csv')

# df.head()

ImportError: cannot import name 'TEAM_NAMES' from 'team_helper' (/Users/zak/Jupyter Notebooks/BaseballAttendance/team_helper.py)

In [2]:
app = Dash()

app.layout = html.Div([
    html.H1(children='Attendance Data by Team and Year', style={'textAlign':'center'}),
    html.Div([
        html.Div(
            dcc.Dropdown(
                [
                    # specify labels/values to make selection box clearer
                    {'label': TEAM_NAMES.get(code, code), 'value': code} for code in sorted(df['home_team'].unique(), key=lambda c: TEAM_NAMES.get(c, c))
                ],
                'DET', id='team-dropdown'
            ),
            style={'width': '70%', 'padding': '10px'}
        ),
        html.Div(
            dcc.Dropdown(sorted(df['year'].unique()), df['year'].max(), id='year-dropdown'),
            style={'width': '30%', 'padding': '10px'}
        ),
    ], style={'display': 'flex', 'flexDirection': 'row'}),

    html.Div([
        html.Div([
            dcc.Graph(id='attendance-by-team-graph', style={'width': '50%'}),
            dcc.Graph(id='attendance-by-opponent-graph', style={'width': '50%'}),
        ], style={'display': 'flex', 'flexDirection': 'row'}),

        html.Div([
            dcc.Graph(id='attendance-by-game-importance-graph', style={'width': '50%'}),
            dcc.Graph(id='attendance-by-day-or-night-graph', style={'width': '50%'}),
        ], style={'display': 'flex', 'flexDirection': 'row'}),
    ], style={'display': 'flex', 'flexDirection': 'column', 'gap': '10px'})
], style={'backgroundColor': '#ffffff', 'color': '#111', 'minHeight': '100vh', 'padding': '20px', 'font-family': 'sans-serif'})

@callback(
    Output('attendance-by-team-graph', 'figure'),
    Output('attendance-by-opponent-graph', 'figure'),
    Output('attendance-by-game-importance-graph', 'figure'),
    Output('attendance-by-day-or-night-graph', 'figure'),
    Input('team-dropdown', 'value'),
    Input('year-dropdown', 'value')
)
def update_graph(team, year):
    # select data by team(s) and year
    dff = df[(df['home_team'] == team ) & (df['year'] == year)]
    
    # year dataframe to set mins/maxs for each year
    year_df = df[df['year'] == year]

    # attendance min/max for axis mins/maxs
    attend_min, attend_max = year_df['attendance'].min(), year_df['attendance'].max()

    # ===============
    # LINE GRAPH
    # attendance by how far into the season the game is
    # ===============

    # plot the line, x is the home game number, y is the attendance
    # set color by the team selected, and map it to the values from `team_helper.py`
    # update the labels
    line_graph = px.line(dff, x='game_number', y='attendance', color='home_team', color_discrete_map=get_team_colors([team]),
                  labels={'game_number': 'Home Game Number', 'attendance': 'Attendance', 'home_team': 'Team'}, title='Attendance Home Game Number')
    
    # set a global min/max for the y axis
    line_graph.update_yaxes(range=[attend_min, attend_max])
    
    # hide legend and center title
    line_graph.update_layout(showlegend=False, title_x=0.5)

    # ===============
    # BAR CHART
    # average attendance by opponent
    # ===============
    dff_grouped = dff.groupby('away_team').agg(avg_attend=('attendance', 'mean')).reset_index()
    dff_grouped = dff_grouped.sort_values('avg_attend', ascending=False)
    bar_chart = px.bar(dff_grouped, y='away_team', x='avg_attend', orientation='h',
                       color='away_team', color_discrete_map=get_team_colors(dff_grouped['away_team']),
                       labels={'away_team': 'Opponent', 'avg_attend': 'Average Attendance'},
                       title='Average Attendance by Opponent')
    
    # get the max average for that year to set all x-axis values for that year to the same values
    max_avg_attendance = year_df.groupby(['home_team', 'away_team'])['attendance'].mean().max()
    # min/max the axes
    bar_chart.update_xaxes(range=[0, max_avg_attendance])
    bar_chart.update_yaxes(tickmode='linear', dtick=1)
    # hide legend and center title
    bar_chart.update_layout(showlegend=False, title_x=0.5)

    # ===============
    # SCATTER PLOT
    # attendance by how important the game was (cLI or championship leverage index)
    # ===============
    
    # size the point to be a percentage of the max attendance on the season
    scatter_plot = px.scatter(dff, x='cLI', y='attendance', color='home_team', color_discrete_map=get_team_colors([team]),
                              labels={'cLI': 'Championship Leverage Index (Game Importance)', 'attendance': 'Attendance'},
                              title='Attendance vs. Game Importance')

    # set min/max on y axes
    scatter_plot.update_yaxes(range=[attend_min, attend_max])

    # hide legend and center title
    scatter_plot.update_layout(showlegend=False, title_x=0.5)

    # ===============
    # JITTER DOT PLOT
    # attendance by whether it's a day or night game
    # ===============
    
    # determine the day of the week
    dff['day_of_week'] = pd.to_datetime(dff['date']).dt.day_name()

    # specify the order to display, otherwise it starts with Friday
    day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

    # jitter plot it
    jittered_dot_plot = px.strip(
        dff, x='day_of_week', y='attendance', color='day_night',
        category_orders={'day_of_week': day_order},                 # use the order above
        color_discrete_map={'D': '#F5A623', 'N': '#1B1F3B'},    # use better day/night colors
        labels={'day_of_week': 'Day of Week', 'attendance': 'Attendance', 'day_night': 'Day/Night'},
        title='Attendance by Day of Week and Game Time'
    )

    # overlay a box plot
    box_overlay = px.box(dff, x='day_of_week', y='attendance', category_orders={'day_of_week': day_order})
    box_overlay.update_traces(fillcolor='rgba(255,255,255,0)', line=dict(color='black', width=1), opacity=0.4, showlegend=False)

    # make sure everything in each part of the jitter plot is aligned correctly
    # also center the title
    jittered_dot_plot.update_layout(boxmode='overlay', title_x=0.5)

    for trace in box_overlay.data:
        jittered_dot_plot.add_trace(trace)

    return line_graph, bar_chart, scatter_plot, jittered_dot_plot

# run in-notebook
# app.run()
# run in stand-alone browser window
app.run(jupyter_mode='tab')

NameError: name 'df' is not defined